# Figure 6 — from a mouse experiment to a human prediction

§1 to §5 establish what π is and where it holds. This section uses it: an optogenetic activation
map from mouse anterior insula is carried through the coupling and read out as a prediction about
human cortex.

The operation is a single weighted average. For a mouse map `v` and coupling `π`:

```
prediction[j] = Σ_i v[i]·π[i,j] / Σ_i π[i,j]
```

Everything else in the figure summarises that vector.

The null is the part worth attention. A smooth mouse map routed through any coupling produces a
smooth human map, so the question is whether it is this coupling that places the signal in
salience cortex. The null permutes π's rows, preserving both maps and destroying only the
correspondence.

Before running: `python scripts/fetch_data.py`, plus the `transbrain` package for §4.

In [ ]:
import json, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGS = ROOT.parent / 'manuscript' / 'figures'
sys.path.insert(0, str(ROOT / 'src'))

from otter.data import load_pi, pi_provenance

pi = load_pi()
PROV = pi_provenance()
LOGS = ROOT / 'outputs' / 'logs'
DATA = ROOT / 'data_external' / 'transbrain_2025'
print(f"coupling {pi.shape[0]:,} x {pi.shape[1]:,}   {PROV['pi_file']}   sha {PROV['pi_sha256'][:16]}...")

# parcel -> mouse structure acronym, and human parcel -> Yeo-17 network
mm = json.loads((ROOT / 'data_external/mouse_sc_meta.json').read_text())
parcel_acr = np.array([mm['structure_acronyms'][i] for i in mm['node_struct_idx']])
nr = np.asarray(json.loads((ROOT / 'data_external/human_sc_meta.json').read_text())['node_region'], int)
rows = [l.split('\t') for l in (ROOT / 'outputs/anndata/_schaefer_order.txt').read_text().splitlines() if l.strip()]
net = np.array([{int(p[0]): p[1].split('_', 2)[2].split('_')[0] for p in rows}.get(int(k), '?') for k in nr])
nets = sorted({u for u in set(net) if u != '?' and (net == u).sum() >= 10})
print(f"{len(nets)} Yeo-17 networks over {len(net):,} human parcels")

PUBLISHED = {
    'SalVentAttnB (SD)':        (1.02, 0.03),
    'VisCent (SD)':             (-1.30, 0.03),
    'salience enrichment (SD)': (0.86, 0.03),
    'salience null p':          (0.001, 0.002),
    'Dvl1 salience (SD)':       (0.15, 0.03),
    'Slc6a4 salience (SD)':     (-0.39, 0.03),
}

def check(name, value):
    exp, tol = PUBLISHED[name]
    ok = abs(value - exp) <= tol
    print(f"  [{'ok ' if ok else 'FAIL'}] {name:26s} computed {value:+.4g}   manuscript {exp}")
    return ok

## 1. The operation (Fig. 6a, 6e)

Route the mouse anterior-insula optogenetic map through π. `route` is the transport-weighted
average written out above; `profile` z-scores the resulting human map and averages within each
Yeo-17 network.

In [ ]:
def route(value_by_acronym, coupling):
    '''Transport-weighted average: prediction[j] = sum_i v[i]*pi[i,j] / sum_i pi[i,j].'''
    v = np.array([value_by_acronym.get(a, np.nan) for a in parcel_acr])
    mask = np.isfinite(v)
    num = v[mask] @ coupling[mask, :]
    den = coupling[mask, :].sum(0)
    out = np.full(coupling.shape[1], np.nan)
    ok = den > 1e-12
    out[ok] = num[ok] / den[ok]
    return out


def profile(human_map):
    '''Network means of the z-scored prediction.'''
    m = np.isfinite(human_map)
    z = (human_map - np.nanmean(human_map[m])) / np.nanstd(human_map[m])
    return {u: float(np.nanmean(z[m & (net == u)])) for u in nets}, z, m


ai = pd.read_csv(DATA / 'ai_opto.csv', index_col=0).iloc[:, 0].to_dict()
pred = route(ai, pi)
prof, z, m = profile(pred)

print(f"routed onto {int(m.sum()):,} human parcels\n")
print('network ranking of the translated map (z):')
for u in sorted(prof, key=prof.get, reverse=True):
    print(f"  {u:16s} {prof[u]:+.2f}")
print()
check('SalVentAttnB (SD)', prof['SalVentAttnB'])
check('VisCent (SD)', prof['VisCent'])

## 2. Is the prediction specific to this coupling? (Fig. 6b)

Salience enrichment is the difference between the salience and ventral-attention networks and the
rest of cortex, in SD units. The null re-routes the same mouse map through a row-permuted π, so
both maps keep their structure and only the mouse→human correspondence is broken.

1,000 permutations, each a full re-route. This is the slow cell, on the order of a couple of
minutes.

In [ ]:
N_PERM = 1000        # lower this for a quick pass; the published p uses 1000

sal = m & np.char.startswith(net.astype(str), 'SalVentAttn')
obs = float(np.nanmean(z[sal]) - np.nanmean(z[m & ~sal]))

rng = np.random.default_rng(0)
t0 = time.time()
null = []
for _ in range(N_PERM):
    hh = route(ai, pi[rng.permutation(pi.shape[0])])
    m2 = np.isfinite(hh)
    zz = (hh - np.nanmean(hh[m2])) / np.nanstd(hh[m2])
    null.append(float(np.nanmean(zz[sal & m2]) - np.nanmean(zz[m & ~sal & m2])))
null = np.array(null)
p = (np.sum(null >= obs) + 1) / (N_PERM + 1)

print(f"{N_PERM} permutations in {time.time() - t0:.0f} s\n")
print(f"salience enrichment  observed {obs:+.3f} SD")
print(f"permuted-pi null     mean {null.mean():+.3f}, 95th pct {np.percentile(null, 95):+.3f}")
print(f"p = {p:.3f}\n")
check('salience enrichment (SD)', obs)
check('salience null p', p)

fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.hist(null, bins=40, color='#c9c9c9', edgecolor='none')
ax.axvline(obs, color='#c1272d', lw=2)
ax.text(obs, ax.get_ylim()[1] * 0.9, f'  observed {obs:+.2f}', color='#c1272d', fontsize=9)
ax.set_xlabel('salience enrichment (SD)'); ax.set_ylabel('permutations')
ax.set_title(f'The prediction is specific to this coupling\np = {p:.3f}',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 3. Five autism models (Fig. 6d)

The same operation applied to cortical atrophy patterns from five mouse autism models. They
implicate different networks rather than differing in severity.

This is a description of model heterogeneity. No null is attached to the between-model
comparison, and the manuscript says so; five models scored on one axis would not support a
ranking.

In [ ]:
mut = pd.read_csv(DATA / 'mouse_mutation_pattern.csv', index_col=0)
autism = {g: profile(route(mut[g].to_dict(), pi))[0] for g in mut.columns}

def sal_enrich(pr):
    s = [v for u, v in pr.items() if u.startswith('SalVentAttn')]
    o = [v for u, v in pr.items() if not u.startswith('SalVentAttn')]
    return float(np.mean(s) - np.mean(o))

enr = {g: sal_enrich(pr) for g, pr in autism.items()}
print('salience enrichment by model (SD):')
for g in sorted(enr, key=enr.get, reverse=True):
    top = max(autism[g], key=autism[g].get)
    print(f"  {g:10s} {enr[g]:+.2f}   peak network: {top}")
print()
check('Dvl1 salience (SD)', enr['Dvl1'])
check('Slc6a4 salience (SD)', enr['Slc6a4'])
print("\nThe models differ in WHICH network they implicate, which is the claim; no null is")
print("attached to the between-model comparison and none should be read into the ordering.")

## 4. Against TransBrain (Fig. 6c)

The same circuit translated by both methods, scored on the same parcels with the same metric and
matched permutation counts. This needs the `transbrain` package; otherwise the head-to-head log
is provenance-checked and read.

In [ ]:
try:
    import transbrain
    HAVE_TB = True
except Exception as e:
    HAVE_TB = False
    print(f"transbrain unavailable ({type(e).__name__}); reading the head-to-head log instead")

RUN_HEADTOHEAD = False      # True re-runs the comparison (needs transbrain; several minutes)

if RUN_HEADTOHEAD and HAVE_TB:
    s = ROOT / 'experiments' / 'transbrain_2025_benchmark' / '05_aiopto_headtohead.py'
    r = subprocess.run([sys.executable, str(s)], cwd=str(ROOT), capture_output=True, text=True)
    print((r.stdout or r.stderr)[-500:])

tb = json.loads((LOGS / 'section6_transbrain_aiopto.json').read_text())
sha = tb.get('pi_sha256')
print('provenance verified' if sha == PROV['pi_sha256'] else f'provenance: {sha}')

# Read the specific fields rather than filtering on a guessed shape -- a loop that matches
# nothing prints nothing, and an empty section looks the same as a passing one.
spec = tb['specificity']
enr_tb = tb['salience_enrichment_z']
print(f"\nscored on the same {tb['shared_support_parcels']:,} parcels "
      f"({tb['n_salience_parcels']} salience, {tb['n_rest_parcels']} rest), "
      f"{spec['otter']['n_perm']} permutations each\n")
for meth in ('otter', 'transbrain'):
    s = spec[meth]
    verdict = 'exceeds its null' if s['p'] < 0.05 else 'does NOT exceed its null'
    print(f"  {meth:11s} enrichment {s['obs']:+.2f} SD   "
          f"null {s['null_mean']:+.2f} +/- {s['null_std']:.2f}   p = {s['p']:.3f}   {verdict}")
print(f"\n  difference {enr_tb['diff_otter_minus_tb']:+.2f} SD in OTTER's favour")

rk = tb['salience_network_rank']
print(f"\nwhere each method ranks the salience networks (of {rk['n_networks']}):")
for meth in ('otter', 'transbrain'):
    r = rk[meth]
    print(f"  {meth:11s} SalVentAttnB #{r['SalVentAttnB']}, SalVentAttnA #{r['SalVentAttnA']}")
print("\nOTTER puts SalVentAttnB first; TransBrain puts it eighth. Both are 'positive' on")
print("salience, but only one localises the circuit to the network the mouse experiment targeted.")

print("\nBoth methods are scored on the same parcels with the same metric and the same number of")
print("permutations. An earlier version compared them at different permutation counts, which is")
print("not a fair test of whether each exceeds its own null.")

## 5. Build the figure panels

In [ ]:
RUN_PANELS = True

if RUN_PANELS:
    r = subprocess.run([sys.executable, str(FIGS / 'fig6' / 'make_fig6_translation.py')],
                       cwd=str(ROOT), capture_output=True, text=True)
    print((r.stdout or r.stderr)[-600:])
    print('ok' if r.returncode == 0 else f'FAILED ({r.returncode})')
else:
    print('skipped; set RUN_PANELS = True')

## 6. Where §6 stands

A mouse optogenetic experiment becomes a specific, falsifiable statement about human cortex:
activate mouse anterior insula and the human correlate should lie in salience and
ventral-attention cortex rather than visual cortex. The prediction is specific to this coupling,
survives a permutation null, and is sharper than the same translation through a transcriptomic
method.

Coverage does not, however, resolve disorders. See §6 of the manuscript for that negative result
and the spin nulls behind it.

In [ ]:
print(f"coupling               {PROV['pi_file']}")
print(f"AI-opto peak network   {max(prof, key=prof.get)} ({max(prof.values()):+.2f} SD)")
print(f"AI-opto lowest         {min(prof, key=prof.get)} ({min(prof.values()):+.2f} SD)")
print(f"salience enrichment    {obs:+.2f} SD, permuted-pi p = {p:.3f} ({N_PERM} permutations)")
print(f"autism models          {min(enr.values()):+.2f} to {max(enr.values()):+.2f} SD salience enrichment")
print()
ok = all([check('SalVentAttnB (SD)', prof['SalVentAttnB']),
          check('VisCent (SD)', prof['VisCent']),
          check('salience enrichment (SD)', obs),
          check('salience null p', p),
          check('Dvl1 salience (SD)', enr['Dvl1']),
          check('Slc6a4 salience (SD)', enr['Slc6a4'])])
print('\nALL CHECKS PASS' if ok else '\nSOME CHECKS FAILED -- text and code have diverged')